# Score model from specific folder

When MCFOST has already finished, so only calculation and figures

In [ ]:

from __future__ import annotations

import argparse
from functools import partial
import json
import math
import os
os.environ["MPLBACKEND"] = "Agg"   # must be set before importing matplotlib
import matplotlib
matplotlib.use("Agg", force=True)

import shutil
import subprocess
import sys
import textwrap
import tempfile
import time
import traceback
from pathlib import Path
from typing import Any, Dict, Tuple
import numpy as np
import matplotlib.pyplot as plt



# --- Cluster imports ---
# pip install dask distributed dask-jobqueue
from distributed import Client
from dask_jobqueue import SLURMCluster

# --- SMAC3 / ConfigSpace ---
# pip install smac configspace
from smac import Scenario
from smac.facade.multi_fidelity_facade import MultiFidelityFacade
from smac.runhistory import TrialValue
from smac import HyperparameterOptimizationFacade as HPO  # for callbacks, utils

from ConfigSpace import ConfigurationSpace, Float, Integer, Categorical, Configuration
from ConfigSpace.conditions import InCondition, EqualsCondition

from smac.runhistory.dataclasses import TrialInfo, TrialValue
from smac.initial_design.sobol_design import SobolInitialDesign




import distroi

sys.path.append(os.path.abspath(".."))  # parent of current working dir
#import lib.Katya_func as kf

from lib.obriy_fidelity import load_fidelity_config, map_budget_to_fidelity, fidelity_products
from lib.obriy_interferometry_data import load_interferometry_data

import lib.obriy_general as obg
import lib.obriy_sed as obs
import lib.obriy_interferometry as obi
import lib.obriy_polarimetry as obp
import lib.obriy_mcfost as obm
import lib.obriy_after_optimisation as oao
import lib.obriy_alma as oba


matplotlib.rcParams["font.family"] = "serif"
matplotlib.rcParams["font.serif"] = [
    "DejaVu Serif",         # shipped with matplotlib
    "Liberation Serif",
    "Nimbus Roman",
    "TeX Gyre Termes",
    "Times",                # generic fallback
]



## Discover the Dubrulle runs

The configured root contains `sed`, `sed_alma`, `sed_interf`, `sed_interf_pdi_alma_full_all`, `sed_interf_pdi_alma_stages` and `sed_pdi`. Run this notebook on the cluster with access to that directory. Discovery reports the exact files used and any missing or ambiguous inputs. Use `RUN_OVERRIDES` for an ambiguous path; no run is chosen by modification date.

A `trials.csv` export is preferred. If it is absent, a single named-row SMAC `runhistory.json` supplies configurations and trial results directly. Timing is taken from the matching history. ConfigSpace and stage files are located separately, by their contents. Saved ConfigSpace JSON and `fidelity_resolved.json` take precedence over input examples.

Runs have different objectives; final total scores are presented separately. Set `comparison_group` in `RUN_OVERRIDES` only for scientifically compatible runs. Identical product names alone do not confirm compatibility. The heatmap compares individual original contributions, with missing terms shown in grey.

Parameter names, bounds, log scales, categories and ordinal sequences are read from each run's ConfigSpace. Optional YAML `meta` labels and units or `PARAMETER_OVERRIDES` improve presentation. The comparison plot uses the union of configured display domains without changing model bounds or scores. Conditional missing values remain missing.

Trial CSV order is not chronological. Convergence requires recorded start/end times; summed runtime is computational effort, not parallel elapsed wall time. Adjacent budgets are automatically selected for paired rank plots; unpaired models are not assigned fabricated ranks.

In [ ]:
RUN_ROOT = Path("/fred/oz061/kandrych/smac/1_akke_2zones_wider_tapered/dubrulle")
RUN_NAMES = ["sed", "sed_alma", "sed_interf", "sed_interf_pdi_alma_full_all",
             "sed_interf_pdi_alma_stages", "sed_pdi"]
# Discovery uses each run's own files. Override ambiguous paths here if needed.
RUN_OVERRIDES = {
    # "sed": {"config_space": "/path/to/config.yaml", "history_json": "/path/to/runhistory.json"},
}
RUNS = None  # None discovers RUN_NAMES; a manual list still works.
OUTPUT_DIR = RUN_ROOT / "after_optimisation_comparison"
TOP_N = 10
# Specify every sampled parameter when distinguishing physical configurations.
# None infers columns from trials.csv; inspect the printed list before interpreting.
PARAMETER_COLUMNS = None
# Optional presentation overrides. Names, ranges, log scales and choices come
# from each run's config_space. Labels and units can be supplied in YAML meta.
PARAMETER_OVERRIDES = {
    # "zone_1_Rc": {"label": "Zone 1 Rc [au]"},
}
# Optional matched rank comparison: [(run_name, lower_budget, higher_budget)]
RANK_PAIRS = None  # None compares adjacent recorded budgets within every run.


In [ ]:
COMPONENT_COLUMNS = ["SED", "OI_H", "OI_K", "OI_L", "OI_N", "PDI_V", "PDI_I", "PDI_H", "ALMA"]
METADATA_COLUMNS = {"config_id", "seed", "budget", "cost", "time", "status", "additional_info",
                    "instance", "starttime", "endtime", "cpu_time"}

def read_settings(path):
    import yaml
    with Path(path).open() as stream:
        return yaml.safe_load(stream)

def unique_candidate(paths, description):
    paths = sorted(set(paths))
    if len(paths) > 1:
        raise ValueError(f"Ambiguous {description}: " + ", ".join(map(str, paths)))
    return str(paths[0]) if paths else None

def discover_runs(root, names, overrides):
    if not root.is_dir():
        raise FileNotFoundError(f"Run root is unavailable: {root}. Run on the cluster or change RUN_ROOT.")
    runs, report = [], []
    for name in names:
        folder = root / name
        custom = overrides.get(name, {})
        try:
            if not folder.is_dir():
                raise FileNotFoundError("Run directory missing")
            preferred_csv = folder / "smac_results" / "trials.csv"
            csv = custom.get("trials_csv") or (str(preferred_csv) if preferred_csv.is_file() else
                   unique_candidate(folder.rglob("trials.csv"), "trials exports"))
            history = custom.get("history_json") or unique_candidate(folder.rglob("runhistory.json"), "run histories")
            if not csv and not history:
                raise FileNotFoundError("Neither trials.csv nor runhistory.json found")
            config = custom.get("config_space")
            if not config:
                saved = [path for path in folder.rglob("configspace.json")
                         if isinstance(read_settings(path), dict) and "hyperparameters" in read_settings(path)]
                config = unique_candidate(saved, "saved ConfigSpaces")
            if not config:
                candidates = []
                # Driver chooses YAML in the working root; exclude per-trial copies.
                for path in list(folder.rglob("*.yaml")) + list(folder.rglob("*.yml")):
                    if "trials" in path.relative_to(folder).parts:
                        continue
                    content = read_settings(path)
                    if isinstance(content, dict) and "hyperparameters" in content:
                        candidates.append(path)
                config = unique_candidate(candidates, "ConfigSpace YAML files")
            if not config:
                raise FileNotFoundError("ConfigSpace YAML/JSON missing; set config_space in RUN_OVERRIDES")
            stage = custom.get("stage_file")
            if not stage:
                preferred_stage = folder / "fidelity_resolved.json"
                stage = str(preferred_stage) if preferred_stage.is_file() else unique_candidate(
                    folder.rglob("fidelity_resolved.json"), "resolved stage files")
            if not stage:
                candidates = []
                for pattern in ("*.txt", "*.yaml", "*.yml"):
                    for path in folder.glob(pattern):
                        try:
                            content = read_settings(path)
                        except Exception:
                            continue
                        if isinstance(content, dict) and "fidelity_stages" in content:
                            candidates.append(path)
                stage = unique_candidate(candidates, "stage inputs")
            spec = dict(name=name, trials_csv=csv, history_json=history, config_space=config,
                        stage_file=stage, comparison_group=custom.get("comparison_group"))
            runs.append(spec)
            report.append(dict(**spec, discovery_status="ready"))
        except (ValueError, FileNotFoundError) as error:
            report.append(dict(name=name, discovery_status=str(error)))
    return runs, pd.DataFrame(report)

def history_frame(path):
    history = json.loads(Path(path).read_text())
    rows = history.get("data", [])
    if not rows or not isinstance(rows[0], dict):
        raise ValueError(f"Expected named SMAC trial rows in {path}; legacy history needs explicit conversion.")
    records = []
    for trial in rows:
        config = history["configs"][str(trial["config_id"])]
        records.append({**config, **trial})
    return pd.DataFrame(records)

def parse_info(value):
    if isinstance(value, dict):
        return value
    if value is None or (not isinstance(value, str) and pd.isna(value)):
        return {}
    for parser in (json.loads, ast.literal_eval):
        try:
            result = parser(value)
            if isinstance(result, dict):
                return result
        except (ValueError, SyntaxError, TypeError):
            pass
    raise ValueError("Cannot parse additional_info; preserve and inspect this export rather than dropping it.")

def read_parameter_specs(path):
    if not path:
        return {}
    import yaml
    with Path(path).open() as stream:
        settings = yaml.safe_load(stream)
    result = {}
    for parameter in settings["hyperparameters"]:
        name = parameter["name"]
        meta = parameter.get("meta") or {}
        if not isinstance(meta, dict):
            meta = {}
        label = meta.get("label", name)
        if meta.get("unit"):
            label += f" [{meta['unit']}]"
        spec = {"label": label, "scale": "log" if parameter.get("log", False) else "linear",
                "fixed": parameter["type"] == "constant"}
        if "choices" in parameter:
            spec["categories"] = parameter["choices"]
        elif "sequence" in parameter:
            spec["categories"] = parameter["sequence"]
        elif "lower" in parameter and "upper" in parameter:
            spec["limits"] = (float(parameter["lower"]), float(parameter["upper"]))
        result[name] = spec
    return result

def combined_parameter_specs(runs):
    grouped = {}
    for run in runs:
        for name, spec in read_parameter_specs(run.get("config_space")).items():
            if not spec["fixed"]:
                grouped.setdefault(name, []).append(spec)
    combined = {}
    for name, definitions in grouped.items():
        override = PARAMETER_OVERRIDES.get(name, {})
        discrete = ["categories" in spec for spec in definitions]
        if len(set(discrete)) > 1 and not any(key in override for key in ("categories", "limits")):
            raise ValueError(f"{name}: continuous/discrete definitions differ; provide a display override.")
        scales = {spec["scale"] for spec in definitions}
        if len(scales) > 1 and "scale" not in override:
            raise ValueError(f"{name}: run log scales differ; set an explicit display scale.")
        merged = {"label": definitions[0]["label"], "scale": definitions[0]["scale"]}
        if all(discrete):
            merged["categories"] = list(dict.fromkeys(value for spec in definitions for value in spec["categories"]))
        elif not any(discrete):
            limits = [spec["limits"] for spec in definitions if "limits" in spec]
            if limits:
                merged["limits"] = (min(bounds[0] for bounds in limits), max(bounds[1] for bounds in limits))
        merged.update(override)
        combined[name] = merged
    for name, override in PARAMETER_OVERRIDES.items():
        combined.setdefault(name, {"label": name, "scale": "linear"}).update(override)
    return combined

def stage_lookup(path):
    if not path:
        return {}
    import yaml
    with Path(path).open() as stream:
        settings = yaml.safe_load(stream)
    stages = settings.get("stages")
    if stages is None:
        eta = settings.get("eta", 3)
        stages = [dict(stage=s["name"], products=s["products"], budget=eta**i)
                  for i, s in enumerate(settings["fidelity_stages"])]
    return {float(s["budget"]): s for s in stages}

def extract_components(info):
    values = dict.fromkeys(COMPONENT_COLUMNS, np.nan)
    values["SED"] = info.get("sed", {}).get("chi2_reduced", np.nan)
    oi = info.get("vis2_chromatic", info.get("vis2_1perband", {}))
    for instrument, column in zip(("pionier", "gravity", "matisse_l", "matisse_n"),
                                  ("OI_H", "OI_K", "OI_L", "OI_N")):
        values[column] = oi.get(instrument, {}).get("chi2_reduced", np.nan)
    for band in ("V", "I", "H"):
        values["PDI_" + band] = info.get("pdi", {}).get(band, {}).get("loss", np.nan)
    values["ALMA"] = info.get("alma", {}).get("loss", np.nan)
    return values

def parameter_key(row, columns):
    # Full sampled parameter tuple, not config_id (which is local to each run).
    return tuple("<inactive>" if pd.isna(row[p]) else repr(row[p]) for p in columns)

def load_run(spec):
    path = Path(spec["trials_csv"] or spec["history_json"])
    if not path.is_file():
        raise FileNotFoundError(f"Trial input does not exist: {path}")
    frame = pd.read_csv(path) if spec.get("trials_csv") else history_frame(path)
    required = {"config_id", "budget", "cost", "status", "time"}
    if required - set(frame):
        raise ValueError(f"Missing columns in {path}: {required - set(frame)}")
    inferred = [c for c in frame if c not in METADATA_COLUMNS]
    configured_parameters = read_parameter_specs(spec.get("config_space"))
    parameters = (PARAMETER_COLUMNS if PARAMETER_COLUMNS is not None else
                  [p for p in configured_parameters if p in frame] if configured_parameters else inferred)
    absent = [p for p, definition in configured_parameters.items() if p not in frame and not definition["fixed"]]
    if absent:
        raise ValueError(f"Configured variable parameters absent from trials CSV: {absent}. Check that this is the run's actual config.")
    if set(parameters) - set(frame):
        raise ValueError(f"Missing requested parameters in {path}: {set(parameters) - set(frame)}")
    infos = [parse_info(v) for v in frame.get("additional_info", pd.Series([None]*len(frame)))]
    stages = stage_lookup(spec.get("stage_file"))
    frame["run"] = spec["name"]
    frame["comparison_group"] = spec.get("comparison_group")
    frame["cost"] = pd.to_numeric(frame["cost"], errors="coerce")
    frame["budget"] = pd.to_numeric(frame["budget"], errors="raise")
    frame["info"] = infos
    frame["config_key"] = [parameter_key(row, parameters) for _, row in frame.iterrows()]
    resolved = [info.get("fidelity") or stages.get(float(budget), {})
                for info, budget in zip(infos, frame["budget"])]
    frame["stage"] = [s.get("stage", s.get("name", "unknown")) for s in resolved]
    frame["products"] = [tuple(sorted(s.get("products", []))) for s in resolved]
    # Unknown products must not be accidentally grouped with another run.
    frame["objective_key"] = [" + ".join(p) if p else f"unknown:{spec['name']}:{b}"
                              for p, b in zip(frame["products"], frame["budget"])]
    frame["stage_key"] = frame["stage"] + " | budget=" + frame["budget"].astype(str)
    if (frame.groupby("budget")["objective_key"].nunique() > 1).any():
        raise ValueError("One budget has conflicting product metadata; separate these histories before ranking.")
    recorded = frame.get("status", pd.Series("unknown", index=frame.index)).astype(str).str.upper()
    # SMAC CSVs may serialize StatusType.SUCCESS, SUCCESS, or numeric enum 1.
    frame["recorded_success"] = recorded.isin(["SUCCESS", "STATUSTYPE.SUCCESS", "1", "1.0"])
    frame["failure_metadata"] = [bool(info.get("failure")) for info in infos]
    frame["finite_cost"] = np.isfinite(frame["cost"])
    frame["penalty"] = frame["cost"] >= 1e99
    frame["usable"] = frame["recorded_success"] & frame["finite_cost"] & ~frame["penalty"] & ~frame["failure_metadata"]
    frame["analysis_status"] = np.select(
        [frame["failure_metadata"], frame["penalty"], ~frame["finite_cost"], frame["usable"]],
        ["failure metadata", "failure penalty", "invalid cost", "usable"], default="other recorded status")
    components = pd.DataFrame([extract_components(info) for info in infos], index=frame.index)
    frame = pd.concat([frame, components], axis=1)
    # CSV export order is not temporal. Enrich via explicit keys when history is supplied.
    history_path = spec.get("history_json")
    if history_path:
        history = json.loads(Path(history_path).read_text())
        rows = history["data"]
        if rows and not isinstance(rows[0], dict):
            raise ValueError("Expected named SMAC runhistory rows; adapt explicitly for this older history format.")
        timing = pd.DataFrame(rows)
        keys = ["config_id", "seed", "budget"]
        if "instance" in frame and "instance" in timing:
            keys.append("instance")
        if not {"starttime", "endtime", *keys} <= set(timing):
            raise ValueError("Runhistory lacks required trial keys or start/end times.")
        if timing.duplicated(keys).any():
            raise ValueError("Ambiguous timing keys; include instance in the trials export.")
        frame = frame.drop(columns=[c for c in ("starttime", "endtime") if c in frame])
        frame = frame.merge(timing[keys + ["starttime", "endtime"]], on=keys, how="left", validate="many_to_one")
    return frame, list(parameters)

def save_figure(fig, name):
    fig.savefig(OUTPUT_DIR / name, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)


## Load and audit runs

Recorded success is required for ranking. A finite score below the failure sentinel and no failure metadata are also required. The worst successful score stays successful. Unknown status encodings are displayed for inspection rather than silently accepted. Missing observable contributions remain missing, never zero.

In [ ]:
if RUNS is None:
    RUNS, discovery_report = discover_runs(RUN_ROOT, RUN_NAMES, RUN_OVERRIDES)
    display(discovery_report)
    if discovery_report["discovery_status"].ne("ready").any():
        raise ValueError("Some requested runs could not be discovered. Resolve the report with RUN_OVERRIDES before analysing all runs.")
    if not RUNS:
        raise ValueError("No runs discovered.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if len({r["name"] for r in RUNS}) != len(RUNS):
    raise ValueError("Run names must be unique.")
PARAMETER_SPECS = combined_parameter_specs(RUNS)
loaded = [load_run(spec) for spec in RUNS]
trials = pd.concat([frame for frame, _ in loaded], ignore_index=True)
parameter_columns = list(dict.fromkeys(p for _, columns in loaded for p in columns))
# Without a config, retain basic plotting support using observed ranges.
for parameter in parameter_columns:
    PARAMETER_SPECS.setdefault(parameter, {"label": parameter, "scale": "linear"})
print("Configuration columns used for distinct model selection:", parameter_columns)
display(trials.groupby(["run", "stage_key", "analysis_status"], dropna=False).size().rename("evaluations").reset_index())
display(trials.groupby("run")["status"].value_counts(dropna=False).rename("recorded_status_count"))
stage_summary = trials.groupby(["run", "stage_key", "objective_key"], dropna=False).agg(
    evaluations=("cost", "size"), usable=("usable", "sum"),
    distinct_configurations=("config_key", "nunique"), summed_runtime_s=("time", "sum"))
display(stage_summary)
stage_summary.to_csv(OUTPUT_DIR / "stage_summary.csv")
successful = trials[trials["usable"]].copy()
if successful.empty:
    raise ValueError("No usable trials. Inspect original status encodings and failure metadata above.")

if RANK_PAIRS is None:
    RANK_PAIRS = []
    for run, subset in trials.groupby("run", sort=False):
        budgets = sorted(subset["budget"].unique())
        RANK_PAIRS.extend((run, lower, higher) for lower, higher in zip(budgets[:-1], budgets[1:]))
print("Budget pairs:", RANK_PAIRS)


## Best distinct parameter sets

Rank distinct configurations by their **mean recorded cost** within each run and stage. Repeated evaluations contribute a repeat count and standard deviation; taking the minimum of repeats would favour models with more repeats or more noise. Components are also averaged over available records and their counts are retained. This is a descriptive aggregation, not a change to the optimisation objective.

Final means the maximum configured budget when a stage file is provided, otherwise the maximum recorded budget (including failures). It is not silently replaced by a lower successful budget.

In [ ]:
group_columns = ["run", "budget", "stage_key", "objective_key", "comparison_group", "config_key"]
aggregations = {"mean_cost": ("cost", "mean"), "cost_std": ("cost", "std"),
                "repeat_count": ("cost", "size"), "config_id": ("config_id", "first")}
aggregations.update({p: (p, "first") for p in parameter_columns})
aggregations.update({c: (c, "mean") for c in COMPONENT_COLUMNS})
aggregations.update({c + "_count": (c, "count") for c in COMPONENT_COLUMNS})
models = successful.groupby(group_columns, dropna=False).agg(**aggregations).reset_index()
models["rank"] = models.groupby(["run", "stage_key"])["mean_cost"].rank(method="min")
best_by_stage = models.sort_values("mean_cost").groupby(["run", "stage_key"], sort=False).head(TOP_N)
final_budget = {}
for spec in RUNS:
    configured = stage_lookup(spec.get("stage_file"))
    final_budget[spec["name"]] = max(configured) if configured else trials.loc[trials["run"] == spec["name"], "budget"].max()
final_models = models[models.apply(lambda row: row["budget"] == final_budget[row["run"]], axis=1)].copy()
best_final = final_models.sort_values("mean_cost").groupby("run", sort=False).head(TOP_N).copy()
for run in final_budget:
    if run not in set(best_final["run"]):
        print(f"{run}: no usable evaluations at final budget {final_budget[run]}; no final model substituted.")
view_columns = ["run", "stage_key", "rank", "config_id", "mean_cost", "cost_std", "repeat_count"] + COMPONENT_COLUMNS + parameter_columns
display(best_final[view_columns])
best_final.drop(columns="config_key").to_csv(OUTPUT_DIR / "best_final_models.csv", index=False)
best_by_stage.drop(columns="config_key").to_csv(OUTPUT_DIR / "best_models_by_stage.csv", index=False)
# Check whether available component contributions reconstruct the original objective.
successful["available_component_sum"] = successful[COMPONENT_COLUMNS].sum(axis=1, min_count=1)
successful["cost_minus_available_components"] = successful["cost"] - successful["available_component_sum"]
display(successful.groupby(["run", "stage_key"])["cost_minus_available_components"].agg(["count", "min", "max"]))


## Observable cost heatmap

Each cell is an original cost contribution, averaged across repeats. Missing cells are grey. Colours use log10(1 + contribution) for legibility; annotated numbers retain the original cost scale. No row normalisation or reweighting is applied. Comparing a column across runs requires compatible data and preprocessing for that observable.

In [ ]:
if not best_final.empty:
    heat = best_final[COMPONENT_COLUMNS].to_numpy(dtype=float)
    if np.any(heat[np.isfinite(heat)] < 0):
        raise ValueError("Negative component score encountered; inspect metric definitions.")
    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad("lightgrey")
    fig, ax = plt.subplots(figsize=(11, max(3, len(best_final)*0.45)))
    im = ax.imshow(np.ma.masked_invalid(np.log1p(heat)/np.log(10)), aspect="auto", cmap=cmap)
    ax.set_xticks(range(len(COMPONENT_COLUMNS)), COMPONENT_COLUMNS, rotation=45, ha="right")
    labels = [f"{r.run} | rank {r.rank:g} | cfg {r.config_id}" for r in best_final.itertuples()]
    ax.set_yticks(range(len(labels)), labels)
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            if np.isfinite(heat[i, j]):
                ax.text(j, i, f"{heat[i,j]:.3g}", ha="center", va="center", fontsize=8,
                        bbox=dict(facecolor="white", alpha=0.65, edgecolor="none"))
    fig.colorbar(im, ax=ax, label="log10(1 + original contribution)")
    ax.set_title("Best final-stage models: individual observable costs")
    save_figure(fig, "observable_cost_heatmap.png")


## Parameter sets across runs

Parallel axes show the best final-stage configurations. Each continuous axis spans its displayed physical domain; the tick labels below list the original units and range. Log axes are transformed only for display. Discrete axes list their categories. Lines connect complete parameter sets, not an inferred probability distribution. Conditional missing values break the lines.

In [ ]:
plot_parameters = [p for p in PARAMETER_SPECS if p in best_final and best_final[p].nunique() > 1]
if plot_parameters and not best_final.empty:
    transformed, labels = [], []
    for p in plot_parameters:
        spec = PARAMETER_SPECS[p]
        values = best_final[p]
        numeric = pd.api.types.is_numeric_dtype(values)
        categories = spec.get("categories")
        if categories is not None or not numeric:
            categories = categories if categories is not None else sorted(values.dropna().unique(), key=str)
            mapping = {value: i/max(1, len(categories)-1) for i, value in enumerate(categories)}
            mapped = values.map(mapping)
            if (values.notna() & mapped.isna()).any():
                raise ValueError(f"{p}: display categories omit sampled values.")
            transformed.append(mapped.to_numpy(dtype=float))
            labels.append(spec.get("label", p) + "\n" + ", ".join(map(str, categories)))
        else:
            raw = values.to_numpy(dtype=float)
            lo, hi = spec.get("limits", (np.nanmin(raw), np.nanmax(raw)))
            if spec.get("scale") == "log":
                if lo <= 0 or np.any(raw[np.isfinite(raw)] <= 0):
                    raise ValueError(f"{p}: logarithmic display requires positive values.")
                raw = np.log10(raw)
                low, high = np.log10([lo, hi])
            else:
                low, high = lo, hi
            transformed.append((raw-low)/(high-low) if high != low else np.zeros_like(raw))
            labels.append(spec.get("label", p) + f"\n{lo:.3g} to {hi:.3g}" +
                          (" (log)" if spec.get("scale") == "log" else ""))
    coordinates = np.column_stack(transformed)
    fig, ax = plt.subplots(figsize=(max(9, 2*len(labels)), 5))
    for i, run in enumerate(best_final["run"].unique()):
        mask = best_final["run"].eq(run).to_numpy()
        for k, row in enumerate(coordinates[mask]):
            ax.plot(range(len(labels)), row, "o-", color=f"C{i%10}", alpha=0.55, label=run if k == 0 else None)
    ax.set_xticks(range(len(labels)), labels)
    ax.set_ylabel("Position within displayed parameter domain")
    ax.legend()
    save_figure(fig, "best_parameter_sets.png")


## Compatible run comparison

Panels compare final-stage score distributions of distinct evaluated configurations only within a user-confirmed comparison group and an identical product set. Different runs sample different regions, so a distribution difference is not automatically evidence that one optimiser is better. Best means the minimum mean cost among evaluated configurations. Missing compatibility information leaves runs separate.

In [ ]:
comparison_summary = final_models.groupby(["comparison_group", "objective_key", "run"], dropna=False).agg(
    distinct_models=("mean_cost", "size"), best_mean_cost=("mean_cost", "min"), median_cost=("mean_cost", "median"))
display(comparison_summary)
comparison_summary.to_csv(OUTPUT_DIR / "run_comparison_summary.csv")
confirmed = final_models[final_models["comparison_group"].notna() & final_models["comparison_group"].ne("") & ~final_models["objective_key"].str.startswith("unknown:")]
for i, ((group, objective), subset) in enumerate(confirmed.groupby(["comparison_group", "objective_key"])):
    names = list(subset["run"].unique())
    if len(names) < 2:
        continue
    fig, ax = plt.subplots(figsize=(max(6, len(names)*1.6), 4))
    ax.boxplot([subset.loc[subset["run"] == name, "mean_cost"] for name in names])
    ax.set_xticks(range(1, len(names)+1), names)
    ax.set_ylabel("Original composite score")
    ax.set_title(f"{group}\n{objective}")
    save_figure(fig, f"compatible_runs_{i}.png")


## Convergence within each run and stage

Chronology comes from recorded completion times, never CSV order or configuration ID. Curves show best successful score so far against completed evaluations (including unsuccessful evaluations) and cumulative recorded trial runtime. Repeated evaluations are visible here as optimiser events; the ranking table separately averages repeats. If timing is unavailable, convergence is skipped rather than fabricated. Curves remain stage-specific because their objectives can differ.

In [ ]:
for run, subset in trials.groupby("run", sort=False):
    if "endtime" not in subset or subset["endtime"].isna().all():
        print(f"{run}: convergence skipped; supply history_json with trial timing.")
        continue
    if subset["endtime"].isna().any():
        print(f"{run}: convergence skipped; some evaluations lack completion times.")
        continue
    ordered = subset.sort_values("endtime").copy()
    ordered["completed_evaluations"] = np.arange(1, len(ordered)+1)
    durations = pd.to_numeric(ordered["time"], errors="coerce")
    complete_durations = durations.notna().all() and (durations >= 0).all()
    if complete_durations:
        ordered["cumulative_runtime_h"] = durations.cumsum()/3600
    fig, axes = plt.subplots(1, 2 if complete_durations else 1, figsize=(11 if complete_durations else 6, 4), squeeze=False)
    for stage, events in ordered.groupby("stage_key", sort=False):
        events = events.copy()
        events["best_so_far"] = events["cost"].where(events["usable"]).cummin().ffill()
        axes[0,0].step(events["completed_evaluations"], events["best_so_far"], where="post", label=stage)
        if complete_durations:
            axes[0,1].step(events["cumulative_runtime_h"], events["best_so_far"], where="post", label=stage)
    axes[0,0].set_xlabel("Completed evaluations across this run")
    if complete_durations:
        axes[0,1].set_xlabel("Summed trial runtime [hours; not wall time]")
    for ax in axes.flat:
        ax.set_ylabel("Best score so far within stage")
        ax.set_title(run)
        ax.legend(fontsize=8)
    save_figure(fig, f"convergence_{list(trials['run'].unique()).index(run)}.png")


## Matched configurations across budgets

Choose entries in `RANK_PAIRS` to connect identical physical configurations evaluated at two budgets within a run. Ranks are recalculated within the paired subset at each stage; unmatched models are not assigned ranks. This asks whether early scoring preserves the ordering of models that were later evaluated. Promotion selection means these pairs are not an unbiased sample of all early-stage models.

In [ ]:
for pair_index, (run, lower, higher) in enumerate(RANK_PAIRS):
    subset = models[models["run"] == run]
    left = subset[subset["budget"] == lower][["config_key", "mean_cost"]]
    right = subset[subset["budget"] == higher][["config_key", "mean_cost"]]
    paired = left.merge(right, on="config_key", suffixes=("_lower", "_higher"), validate="one_to_one")
    if len(paired) < 2:
        print(f"{run}, {lower} → {higher}: fewer than two matched models.")
        continue
    paired["rank_lower"] = paired["mean_cost_lower"].rank(method="average")
    paired["rank_higher"] = paired["mean_cost_higher"].rank(method="average")
    correlation = paired["rank_lower"].corr(paired["rank_higher"])
    print(f"{run}: {len(paired)} matched models; paired rank correlation = {correlation:.3g}")
    display(paired.drop(columns="config_key"))
    fig, ax = plt.subplots(figsize=(6, 4))
    for row in paired.itertuples():
        ax.plot([0,1], [row.rank_lower, row.rank_higher], "o-", alpha=0.5)
    ax.set_xticks([0,1], [f"Budget {lower}", f"Budget {higher}"])
    ax.set_ylabel("Rank among matched models; 1 is best")
    ax.invert_yaxis()
    ax.set_title(run)
    save_figure(fig, f"paired_stage_ranks_{pair_index}.png")


## Interpretation and outputs

Exported tables retain original score scales and repeat counts. Figure colour transformations and parameter-axis transformations affect display only. Missing components or unknown stages require inspection of the run metadata. A nonzero cost-minus-components diagnostic can indicate incomplete metadata or a different historical score definition; do not repair it by renormalising terms.

For a paper, start with `best_final_models.csv`, `observable_cost_heatmap.png`, and stage-specific convergence figures where timing exists. Parameter plots show the actual best sampled combinations. Use common final-stage reevaluations for a definitive comparison when runs use different final objectives; this notebook does not launch those simulations.

The original notebook remains available for its additional exploratory plots. This copy replaces its status rewriting, double quantile filtering and inverse-cost corner weighting with explicit selection and descriptive comparisons.